<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/Notebook01_Setup_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Boilerplate: Drive mount, working directory, GPU verification ===
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/MyDissertationCN6000'
os.chdir(PROJECT_ROOT)

import torch
assert torch.cuda.is_available(), (
    "CUDA not available. Go to Runtime > Change runtime type and select an A100 or L4 GPU. "
    "If already set, reinstall PyTorch with CUDA: "
    "!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [2]:
from IPython.core.display import json
import urllib.request, zipfile, random
from pathlib import Path

In [3]:
#non-copyright related prompts from MS-COCO val2017

zip_path = Path("/content/annotations_trainval2017.zip")
if not zip_path.exists():
  urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  with zip_ref.open("annotations/captions_val2017.json") as f:
    coco=json.load(f)

In [4]:
#one caption per image to avoid near duplicates
seen_images, captions = set(), []
for annotations in coco["annotations"]:
  if annotations["image_id"] not in seen_images:
    seen_images.add(annotations["image_id"])
    captions.append(annotations["caption"].strip().rstrip("."))

In [5]:
#exclude keywords that may cause overlap with the erased concept

banned=["ghibli", "hayao miyazaki", "miyazaki hayao", "totoro", "howl's moving castle",
        "princess mononoke", "spirited away", "ponyo", "kiki's delivery service",
        "the boy and the heron", "studio ghibli", "ghibli film", "ni no kuni"]
captions=[c for c in captions
          if not any(b in c.lower() for b in banned)]

In [6]:
#deterministic sample
random.seed(2026)
unrelated_prompts = random.sample(captions, 50)

In [7]:
assert len(unrelated_prompts)==50
Path("prompts/unrelated_prompts.json").write_text(json.dumps(unrelated_prompts, indent=2))
print(f"Sampled{len(unrelated_prompts)} captions from COCO val2017 (filtered set: {len(captions)})")
print("First 3: ", unrelated_prompts[:3])

Sampled50 captions from COCO val2017 (filtered set: 5000)
First 3:  ['The picture of three buses on a lot', 'A man skiing down a snow covered slope', 'Two women standing next to each other in a train station']
